# INFOSCI 301 · Interaction Design Companion
## Between Water & Mountains / 水山之间

This executable tutorial rebuilds the atlas's interaction patterns from its versioned teaching snapshots. It follows one rule: **a control earns its place only when it changes what a person can notice, compare, test, explain, or question.**

Learning goals:
1. orient Kunshan and the Mandara study region without implying a historical connection;
2. implement selection, parameter change, overview + detail, brushing/linking, filtering, aggregation, and annotation;
3. distinguish observed records, derived summaries, inferred model patterns, and schematic teaching marks;
4. connect Munzner's nested-model questions to recent visualization-authoring precedents.

In [ ]:
%pip -q install altair plotly

import json, math, requests
import pandas as pd
import altair as alt
import plotly.graph_objects as go
from IPython.display import display, Markdown

alt.data_transformers.disable_max_rows()
BASE = 'https://raw.githubusercontent.com/sunshineluyao/interactive-demo-between-water-and-mountains/main/'

def load_json(path):
    response = requests.get(BASE + path, timeout=30)
    response.raise_for_status()
    return response.json()

waterways = load_json('public/data/kunshan-waterways.geojson')
rainfall = load_json('public/data/kunshan-precipitation.json')
mandara = load_json('public/data/mandara-teaching.json')
print(f"Loaded {len(waterways['features'])} waterways, {len(rainfall['records'])} monthly climate records, and {len(mandara['culture_periods'])} primary CP labels.")

## 1 · Read the evidence boundary before the chart

| Source | Status | What it supports | What it does **not** support |
|---|---|---|---|
| OpenStreetMap waterways | observed/mapped snapshot | spatial context and tagged waterway geometry | flow, quality, completeness, lived meaning |
| NASA POWER precipitation | modeled environmental record | monthly precipitation context at one coordinate | a local gauge, flood history, canal discharge |
| tDAR ceramic record | observed archaeological records | sites, units, levels, decoration observations | direct access to identity or motives |
| CP1–CP14 teaching layer | inferred + transcribed summary | model-pattern orientation reported by the paper | named cultures, peoples, chapters, ranks, or a simple timeline |
| globe arc and orbiting marks | schematic | the course's comparison and source types | migration, causality, or measured paths |

In [ ]:
# INTERACTION: navigate + select on an orthographic globe
places = pd.DataFrame([
    {'place': 'Kunshan / 昆山', 'lat': 31.39, 'lon': 120.98, 'status': 'city coordinate'},
    {'place': 'Mandara study region / 曼达拉山脉', 'lat': 11.0, 'lon': 14.0, 'status': 'approximate regional orientation'},
])

fig = go.Figure()
fig.add_trace(go.Scattergeo(
    lon=places.lon, lat=places.lat, text=places.place + '<br>' + places.status,
    mode='markers+text', textposition=['top left', 'bottom right'],
    marker=dict(size=12, color=['#69d9df', '#ff8a5c'], line=dict(color='white', width=1)),
    hovertemplate='%{text}<extra></extra>', name='places'
))
fig.add_trace(go.Scattergeo(
    lon=[120.98, 14.0], lat=[31.39, 11.0], mode='lines',
    line=dict(color='#f2c078', width=2, dash='dot'),
    hoverinfo='skip', name='question bridge (schematic)'
))
fig.update_geos(
    projection_type='orthographic', projection_rotation=dict(lon=68, lat=20),
    showland=True, landcolor='#dce8eb', showocean=True, oceancolor='#153f51',
    showcountries=True, countrycolor='#81959d', bgcolor='#071d25'
)
fig.update_layout(title='Rotate to orient · the arc is a course question, not a historical route', height=620, margin=dict(l=0,r=0,t=60,b=0), paper_bgcolor='#071d25', font_color='white')
fig.show()

## 2 · What does CP1…CP14 mean?

In O'Brien, Lin, and MacEachern (2015), a **culture period (CP)** is a latent statistical component: a consistent distribution of exterior pottery-decoration types inferred by a Dirichlet-process mixture model. The numerals are identifiers. They are **not** chapter numbers, named cultures, ethnic groups, ranks, dates, or automatically ordered time periods. The paper reports fourteen primary components and pools additional clusters as CP15.

The term *culture painting* names the visualization of inferred component memberships across sites, excavation units, and 10 cm levels. It should not be read as a literal painting of peoples or identity.

In [ ]:
# INTERACTION: select + highlight + details on demand
cp_rows = []
for signature in mandara['signatures']:
    for cp, value in zip(mandara['culture_periods'], signature['values']):
        status = 'dominant reported' if value == 1 else ('smaller reported' if value > 0 else 'not asserted in teaching summary')
        cp_rows.append({'group': signature['group'], 'sites': signature['sites'], 'CP': cp, 'value': value, 'status': status})
cp = pd.DataFrame(cp_rows)

pick = alt.selection_point(fields=['CP'], on='click', clear='dblclick', name='Choose_CP')
base = alt.Chart(cp).mark_rect(stroke='#f3f5f3').encode(
    x=alt.X('CP:N', sort=mandara['culture_periods'], title='Model identifier (not chronology)'),
    y=alt.Y('group:N', title=None),
    color=alt.Color('value:Q', scale=alt.Scale(domain=[0,1], range=['#e0e8e9','#123f52']), legend=None),
    opacity=alt.condition(pick, alt.value(1), alt.value(.24)),
    tooltip=['group', 'sites', 'CP', 'status']
).add_params(pick).properties(width=700, height=150, title='Click a CP label; double-click to clear')
base

In [ ]:
# INTERACTION: change a parameter while keeping scales and units stable
rain = pd.DataFrame(rainfall['records'])
rain['date'] = pd.to_datetime(dict(year=rain.year, month=rain.month, day=1))
chosen_month = alt.param(name='Month', value=1, bind=alt.binding_range(min=1, max=12, step=1, name='Month: '))

monthly = alt.Chart(rain).mark_line(point=True).encode(
    x=alt.X('year:O', title='Year'),
    y=alt.Y('mm_per_day:Q', title='Monthly mean precipitation (mm/day)', scale=alt.Scale(zero=True)),
    tooltip=['year', 'month', alt.Tooltip('mm_per_day:Q', format='.2f')]
).add_params(chosen_month).transform_filter(alt.datum.month == chosen_month).properties(width=720, height=280, title='Move one control; preserve source, units, and y-scale')
monthly

In [ ]:
# INTERACTION: overview + detail, brushing, and linked filtering
brush = alt.selection_interval(encodings=['x'], name='Time_window')
overview = alt.Chart(rain).mark_area(color='#7eaabc').encode(
    x=alt.X('date:T', title=None), y=alt.Y('mm_per_day:Q', title='mm/day')
).add_params(brush).properties(width=720, height=100, title='Drag a time window')

detail = alt.Chart(rain).mark_line(color='#245c70').encode(
    x=alt.X('date:T', title='Selected time'), y=alt.Y('mm_per_day:Q', title='mm/day'),
    tooltip=[alt.Tooltip('date:T', title='Month'), alt.Tooltip('mm_per_day:Q', format='.2f')]
).transform_filter(brush).properties(width=720, height=260)

overview & detail

In [ ]:
# INTERACTION: filter + aggregate, with the denominator visible
waterway_types = pd.DataFrame([f['properties'] for f in waterways['features']])
counts = waterway_types.groupby('waterway', dropna=False).size().rename('mapped_ways').reset_index()
counts['share_of_snapshot'] = counts.mapped_ways / counts.mapped_ways.sum()

alt.Chart(counts).mark_bar(color='#245c70').encode(
    x=alt.X('mapped_ways:Q', title=f"Mapped ways (denominator = {len(waterways['features'])})"),
    y=alt.Y('waterway:N', sort='-x', title='OSM waterway tag'),
    tooltip=['waterway', 'mapped_ways', alt.Tooltip('share_of_snapshot:Q', format='.1%')]
).properties(width=620, height=180, title='A derived count of mapped features—not hydrological volume')

## 3 · A three-choice-per-layer design path

A decision tree should reduce ambiguity, not manufacture certainty. Choose a question, evidence class, task, and interaction; then validate the recommendation at Munzner's domain/community, data/task, idiom, and algorithm levels.

In [ ]:
def design_path(question, evidence, task, interaction):
    allowed = {
        'question': ['locate + orient', 'trace change', 'invite interpretation'],
        'evidence': ['space + time', 'relations', 'community + qualitative'],
        'task': ['explore', 'compare', 'explain + communicate'],
        'interaction': ['select + details', 'filter + change', 'coordinate + annotate'],
    }
    selections = dict(question=question, evidence=evidence, task=task, interaction=interaction)
    for layer, choice in selections.items():
        if choice not in allowed[layer]:
            raise ValueError(f"{layer} must be one of: {allowed[layer]}")
    tests = {
        'select + details': 'Can a keyboard user identify the selected record and recover the overview?',
        'filter + change': 'Can a reader state what changed and what stayed fixed?',
        'coordinate + annotate': 'Can another person trace the annotation to the same evidence?',
    }
    return pd.Series({
        'path': ' → '.join(selections.values()),
        'recommended interaction': interaction,
        'next test': tests[interaction],
        'validation': 'domain/community → data/task → idiom → algorithm',
    })

design_path('trace change', 'space + time', 'compare', 'filter + change')

## Interaction crosswalk

| Vocabulary / precedent | Notebook implementation | Learning change |
|---|---|---|
| Navigate | orthographic globe | locate and reorient evidence |
| Select + highlight + details | CP matrix selection | focus one identifier without losing context |
| Change a parameter | month slider | compare a chosen month on stable units/scales |
| Juxtapose + coordinate | overview/detail views | carry one time window across views |
| Filter + aggregate | OSM tag counts | reduce records while exposing the denominator |
| Author + annotate (DataToon precedent) | evidence-card widget below | preserve observation, interpretation and limitation separately |
| Demonstration/mixed initiative (Falx; Data Formulator) | bounded design helper | inspect a recommendation and revise it; no AI agent is claimed |

In [ ]:
# INTERACTION: author + annotate + inspectable output
import ipywidgets as widgets
from IPython.display import JSON, clear_output

observed = widgets.Textarea(description='Observed:', placeholder='What does the source actually show?')
interpreted = widgets.Textarea(description='Interpreted:', placeholder='What might it mean?')
boundary = widgets.Textarea(description='Boundary:', placeholder='What can this evidence not answer?')
make_card = widgets.Button(description='Build evidence card', button_style='info')
card_output = widgets.Output()

def render_card(_):
    with card_output:
        clear_output()
        display(JSON({'observed': observed.value, 'interpreted': interpreted.value, 'evidence_boundary': boundary.value}))

make_card.on_click(render_card)
display(widgets.VBox([observed, interpreted, boundary, make_card, card_output]))

## 4 · Studio exercise

For one interaction above, complete this five-line script:

1. The person needs to **_____**.
2. They **_____** the control.
3. The system responds by **_____**.
4. This reveals **_____**, while the source cannot answer **_____**.
5. They can reset, recover, or challenge the interpretation by **_____**.

Then ask someone from another discipline to perform the task without your explanation. Record what they misunderstood and revise the design.

## References and precedents

- Munzner, T. (2014). *Visualization Analysis and Design*. CRC Press. https://doi.org/10.1201/b17511
- Munzner, T. *Interactive Views (Chapters 11–12)*. https://www.cs.ubc.ca/~tmm/talks/vad/VAD-interact.pdf
- Satyanarayan, A., Moritz, D., Wongsuphasawat, K., & Heer, J. (2017). Vega-Lite. https://doi.org/10.1109/TVCG.2016.2599030
- Kim, N. W., et al. (2019). DataToon. https://doi.org/10.1145/3290605.3300335
- Wang, C., et al. (2021). Falx. https://doi.org/10.1145/3411764.3445249
- Wang, C., Thompson, J., & Lee, B. (2024). Data Formulator. https://doi.org/10.1109/TVCG.2023.3326585
- O'Brien, J. D., Lin, K., & MacEachern, S. (2015). *Mixture model of pottery distributions…* https://doi.org/10.48550/arXiv.1511.05185
- Ceramic dataset at tDAR. https://doi.org/10.6067/XCV83F4R7D

The repository's in-app reference library supplies full author lists, media credits, and data-source notes.